In [ ]:
# Install the required libraries (uncomment the line below to run in your environment)
%pip install tensorflow matplotlib seaborn numpy

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.utils import to_categorical
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

# Check for GPU and set memory growth to avoid allocation errors
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"Using GPU: {gpus[0]}")
    except RuntimeError as e:
        print(e)
else:
    print("Using CPU for training")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# LOAD AND PREPARE CIFAR-10 DATASET
# ═══════════════════════════════════════════════════════════════════

print("Loading CIFAR-10 dataset...")
(X_train, y_train), (X_test, y_test) = cifar10.load_data()

# Normalize pixel values to [0, 1]
X_train = X_train.astype('float32') / 255.0
X_test = X_test.astype('float32') / 255.0

# Convert labels to categorical (one-hot encoding)
y_train_cat = to_categorical(y_train, 10)
y_test_cat = to_categorical(y_test, 10)

print(f"✓ Training set shape: {X_train.shape}")
print(f"✓ Test set shape: {X_test.shape}")
print(f"✓ Number of classes: {y_train_cat.shape[1]}")

# CIFAR-10 class names
class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# BUILD CONVOLUTIONAL NEURAL NETWORK (CNN) MODEL
# ═══════════════════════════════════════════════════════════════════

print("\nBuilding CNN model...")
model = models.Sequential([
    # Block 1
    layers.Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=(32, 32, 3)),
    layers.BatchNormalization(),
    layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),
    
    # Block 2
    layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),
    
    # Block 3
    layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),
    
    # Fully Connected Layers
    layers.Flatten(),
    layers.Dense(512, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.5),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(10, activation='softmax')
])

# Compile the model
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("✓ Model architecture:")
model.summary()

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# TRAIN THE CNN MODEL
# ═══════════════════════════════════════════════════════════════════

print("\nTraining CNN model on CIFAR-10...")
history = model.fit(
    X_train, y_train_cat,
    batch_size=128,
    epochs=25,
    validation_split=0.1,
    verbose=1
)

print("\n✓ Training completed successfully!")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# VISUALIZE TRAINING HISTORY
# ═══════════════════════════════════════════════════════════════════

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Accuracy plot
axes[0].plot(history.history['accuracy'], label='Training Accuracy', linewidth=2)
axes[0].plot(history.history['val_accuracy'], label='Validation Accuracy', linewidth=2)
axes[0].set_title('Model Accuracy Over Epochs', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Loss plot
axes[1].plot(history.history['loss'], label='Training Loss', linewidth=2)
axes[1].plot(history.history['val_loss'], label='Validation Loss', linewidth=2)
axes[1].set_title('Model Loss Over Epochs', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("✓ Training curves plotted successfully!")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# EVALUATE MODEL ON TEST SET
# ═══════════════════════════════════════════════════════════════════

print("\nEvaluating model on test dataset...")
test_loss, test_accuracy = model.evaluate(X_test, y_test_cat, verbose=0)

print(f"\n{'='*50}")
print(f"TEST SET PERFORMANCE")
print(f"{'='*50}")
print(f"Test Loss:     {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")
print(f"{'='*50}")

# Get predictions
y_pred_probs = model.predict(X_test, verbose=0)
y_pred = np.argmax(y_pred_probs, axis=1)
y_test_labels = np.argmax(y_test_cat, axis=1)

# Detailed classification report
print("\n✓ Classification Report:")
print(classification_report(y_test_labels, y_pred, target_names=class_names, zero_division=0))

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CONFUSION MATRIX VISUALIZATION
# ═══════════════════════════════════════════════════════════════════

cm = confusion_matrix(y_test_labels, y_pred)

plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=class_names, yticklabels=class_names, 
            cbar_kws={'label': 'Count'})
plt.title('Confusion Matrix - CIFAR-10 Image Classification', fontsize=14, fontweight='bold')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

print("✓ Confusion matrix plotted successfully!")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# PER-CLASS PERFORMANCE METRICS
# ═══════════════════════════════════════════════════════════════════

from sklearn.metrics import precision_recall_fscore_support

precision, recall, f1, support = precision_recall_fscore_support(
    y_test_labels, y_pred, average=None, zero_division=0
)

# Create visualization
fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(class_names))
width = 0.25

ax.bar(x - width, precision, width, label='Precision', alpha=0.8)
ax.bar(x, recall, width, label='Recall', alpha=0.8)
ax.bar(x + width, f1, width, label='F1-Score', alpha=0.8)

ax.set_xlabel('Class', fontsize=11, fontweight='bold')
ax.set_ylabel('Score', fontsize=11, fontweight='bold')
ax.set_title('Per-Class Performance Metrics', fontsize=13, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(class_names, rotation=45, ha='right')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
ax.set_ylim(0, 1)

plt.tight_layout()
plt.show()

print("✓ Per-class metrics visualization completed!")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# SAMPLE PREDICTIONS VISUALIZATION
# ═══════════════════════════════════════════════════════════════════

# Select 12 random test images
indices = np.random.choice(len(X_test), 12, replace=False)

fig, axes = plt.subplots(3, 4, figsize=(14, 10))
axes = axes.ravel()

for i, idx in enumerate(indices):
    image = X_test[idx]
    true_label = class_names[y_test_labels[idx]]
    pred_label = class_names[y_pred[idx]]
    confidence = y_pred_probs[idx].max() * 100
    
    # Plot image
    axes[i].imshow(image)
    
    # Color: green if correct, red if incorrect
    color = 'green' if y_test_labels[idx] == y_pred[idx] else 'red'
    
    axes[i].set_title(f'True: {true_label}\nPred: {pred_label} ({confidence:.1f}%)', 
                     color=color, fontweight='bold', fontsize=9)
    axes[i].axis('off')

plt.suptitle('Sample Predictions (Green=Correct, Red=Incorrect)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("✓ Sample predictions displayed!")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# PROJECT SUMMARY
# ═══════════════════════════════════════════════════════════════════

print("\n" + "="*60)
print(" "*15 + "CNN IMAGE CLASSIFICATION - PROJECT SUMMARY")
print("="*60)
print("\n📊 DATASET INFORMATION:")
print(f"   • Dataset: CIFAR-10 (10 object classes)")
print(f"   • Training samples: {len(X_train)}")
print(f"   • Test samples: {len(X_test)}")
print(f"   • Image size: 32x32x3")
print(f"   • Classes: {', '.join(class_names)}")

print("\n🏗️  MODEL ARCHITECTURE:")
print(f"   • Type: Convolutional Neural Network (CNN)")
print(f"   • Convolutional blocks: 3")
print(f"   • Filters progression: 32→64→128")
print(f"   • Total parameters: {model.count_params():,}")

print("\n📈 TRAINING CONFIGURATION:")
print(f"   • Optimizer: Adam")
print(f"   • Learning rate: 0.001")
print(f"   • Epochs: 25")
print(f"   • Batch size: 128")
print(f"   • Regularization: Dropout, Batch Normalization")

print("\n✅ PERFORMANCE METRICS:")
print(f"   • Test Accuracy: {test_accuracy*100:.2f}%")
print(f"   • Test Loss: {test_loss:.4f}")
print(f"   • Best validation accuracy: {max(history.history['val_accuracy'])*100:.2f}%")

print("\n✨ KEY FEATURES:")
print(f"   ✓ Fully functional CNN model trained on CIFAR-10")
print(f"   ✓ Comprehensive evaluation on test dataset")
print(f"   ✓ Training history visualization")
print(f"   ✓ Confusion matrix analysis")
print(f"   ✓ Per-class performance metrics")
print(f"   ✓ Sample predictions with confidence scores")

print("\n" + "="*60)
print("✨ PROJECT COMPLETED SUCCESSFULLY! ✨")
print("="*60 + "\n")